Passo 1: Carregue o CSV

In [1]:
import pandas as pd      # biblioteca de análise de dados
import re                # expressões regulares (para limpar texto)
from datetime import datetime  # para trabalhar com datas

# Confirmar que as importações funcionaram
print('✓ pandas versão:', pd.__version__)
print('✓ Bibliotecas importadas com sucesso!')

print()
print()

CAMINHO = r"C:\Users\leogo\OneDrive\Documentos\SCTEC\GitHub\turma-visualizacao-de-dados\alunos\Leo_Gobel\Semana_04\base_rh.csv"

df = pd.read_csv(
    CAMINHO,
    sep=';',           # separador de colunas
    encoding='cp1252', # encoding do Windows
    decimal=','        # vírgula como separador decimal
)

✓ pandas versão: 3.0.1
✓ Bibliotecas importadas com sucesso!




Passo 2: Diagnóstico


In [3]:

df.info()

df.describe().round(2)




<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Funcionario  1000 non-null   int64  
 1   Nome            1000 non-null   str    
 2   Departamento    1000 non-null   str    
 3   Cargo           1000 non-null   str    
 4   Salario         1000 non-null   float64
 5   Data_Admissao   1000 non-null   str    
 6   Genero          1000 non-null   str    
 7   Idade           1000 non-null   int64  
 8   Estado_Civil    1000 non-null   str    
 9   Status          1000 non-null   str    
dtypes: float64(1), int64(2), str(7)
memory usage: 78.3 KB


,ID_Funcionario,Salario,Idade
count,1000.00,1000.00,1000.00
mean,500.50,8579.95,41.40
std,288.82,3657.37,13.67
min,1.00,2000.71,18.00
25%,250.75,5564.54,30.00
50%,500.50,8571.13,41.00
75%,750.25,11554.63,53.00
max,1000.00,14954.51,65.00


Passo 3: Converter Datas e Criar Colunas Derivadas


In [10]:
print(f'Tipo ANTES : {df["Data_Admissao"].dtype}')
print(f'Valor antes: "{df["Data_Admissao"].iloc[0]}"')
# .iloc[0] → acessa a primeira linha pelo índice (iloc = integer location)


# Converter a coluna inteira
# Trabalhamos em uma cópia para não alterar o df original
df_limpo = df.copy()
# .copy() → cria uma cópia independente
# Sem isso, mudanças em df_limpo afetariam o df original

df_limpo['Data_Admissao'] = pd.to_datetime(
    df_limpo['Data_Admissao'],
    format='%d/%m/%Y',  # formato da string de entrada
    errors='coerce'     # datas inválidas viram NaT (Not a Time) em vez de erro
)

print(f'Tipo DEPOIS: {df_limpo["Data_Admissao"].dtype}')
print(f'Valor depois: {df_limpo["Data_Admissao"].iloc[0]}')
print(f'NaT gerados : {df_limpo["Data_Admissao"].isna().sum()}')


# Criar colunas derivadas de data
df_limpo['Ano_Admissao'] = df_limpo['Data_Admissao'].dt.year
df_limpo['Mes_Admissao'] = df_limpo['Data_Admissao'].dt.month

# Calcular tempo de casa em anos
# pd.Timestamp.today() → data atual no formato do pandas
hoje_ts = pd.Timestamp.today()

df_limpo['Tempo_Casa_Anos'] = (
    (hoje_ts - df_limpo['Data_Admissao']).dt.days / 365.25
).round(1)
# Subtração de datas → timedelta
# .dt.days → extrai a diferença em dias
# / 365.25 → converte dias em anos
# .round(1) → arredonda para 1 casa decimal

# Ver o resultado
cols = ['Nome', 'Data_Admissao', 'Mes_Admissao', 'Tempo_Casa_Anos']
df_limpo[cols].head(6)

Tipo ANTES : str
Valor antes: "13/08/2024"
Tipo DEPOIS: datetime64[us]
Valor depois: 2024-08-13 00:00:00
NaT gerados : 0


,Nome,Data_Admissao,Mes_Admissao,Tempo_Casa_Anos
0,Julia Nunes,2024-08-13,8,1.7
1,Sr. Gustavo Duarte,2017-04-29,4,9.0
2,Srta. Mariana Cunha,2024-12-11,12,1.4
3,Ana Sophia da Cruz,2019-06-16,6,6.9
4,Dr. Nicolas Pinto,2019-03-29,3,7.1
5,Marcela Costa,2024-07-23,7,1.8


In [11]:
def classificar_tempo(anos):
    """
    Recebe: anos (número) — o tempo de casa do funcionário
    Retorna: uma categoria em texto
    """
    # pd.isna() verifica se o valor é nulo (NaN ou NaT)
    if pd.isna(anos):
        return 'Sem data'
    elif anos < 1:
        return 'Menos de 1 ano'
    elif anos < 3:
        return '1 a 3 anos'
    elif anos < 5:
        return '3 a 5 anos'
    else:
        return 'Mais de 5 anos'
    
df_limpo['Faixa_Tempo_Casa'] = df_limpo['Tempo_Casa_Anos'].apply(classificar_tempo)
print('Distribuição por faixa de tempo de casa:')


Distribuição por faixa de tempo de casa:


Passo 4: groupby — Maior e Menor Média de Tempo de Casa

In [8]:
# Tempo médio de casa por departamento
media_depto = (
    df_limpo
    .groupby('Departamento')['Tempo_Casa_Anos']
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .rename('Media_Anos')   # .rename() → renomeia a coluna resultante
    .reset_index()          # .reset_index() → converte o índice em coluna comum
)

print('Tempo médio de casa por departamento:')
print(media_depto.to_string(index=False))

# .idxmax() → retorna o NOME do índice com maior valor
maior = media_depto.set_index('Departamento')['Media_Anos'].idxmax()
menor = media_depto.set_index('Departamento')['Media_Anos'].idxmin()
print(f'\n→ Maior retenção: {maior}')
print(f'→ Menor retenção: {menor}')


Tempo médio de casa por departamento:
Departamento  Media_Anos
          RH         5.8
      Vendas         5.7
    Produção         5.5
          TI         5.5
   Logística         5.3
  Financeiro         5.1

→ Maior retenção: RH
→ Menor retenção: Financeiro


Passo 5: Exportar para Excel

Exporte `base_rh_deptos.xlsx` com uma aba `'Completo'` e mais uma aba por departamento.

In [ ]:
%pip install openpyxl

# Exportação avançada: uma aba por departamento
with pd.ExcelWriter('base_rh_deptos.xlsx', engine='openpyxl') as writer:

    # Aba 1: dataset completo
    df.to_excel(writer, sheet_name='Completo', index=False)
    print('  Aba Completo: todos os registros')

    # df['Departamento'].unique() → lista de departamentos sem repetição
    # sorted() → ordena em ordem alfabética
    for depto in sorted(df['Departamento'].unique()):

        # Filtrar só os funcionários desse departamento
        df_depto = df[df['Departamento'] == depto]

        # Limite do Excel: nomes de aba têm no máximo 31 caracteres
        nome_aba = depto[:31]

        df_depto.to_excel(writer, sheet_name=nome_aba, index=False)
        print(f'  Aba "{nome_aba}": {len(df_depto)} registros')

print('\n✓ base_rh_deptos.xlsx gerado!')


Passo 6: cole o output do git log aqui como comentário

In [ ]:
#git pull origin master
#git add base_rh_deptos.xlsx
#git commit -m "semana 04 - dia 01: entrega arquivos e datetime"
#git push origin master
#git log --oneline -3

git log:

In [ ]:
# a188f71 (HEAD -> master, origin/master, origin/HEAD) semana 04 - dia 01: entrega arquivos e datetime
# 2388144 Merge branch 'master' of https://github.com/cfneves/turma-visualizacao-de-dados
# ee4323d Adiciona arquivos semana 04